In [ ]:
import json
import os
import glob
from pathlib import Path
import yaml
from functools import cache, reduce
import shutil

import tqdm

with open("../data_dir.json", "r") as f:
    json_data = json.load(f)
data_dir = json_data["ad_data_dir"]
config_dir = json_data["config_dir"]
if not data_dir:
    data_dir = os.path.join(os.path.dirname(os.getcwd()), "data", "ad-datasets")
else:
    data_dir = os.path.expanduser(data_dir)
if not config_dir:
    config_dir = os.path.join(os.path.dirname(os.getcwd()), "configs")
else:
    config_dir = os.path.expanduser(config_dir)

def abs_config_path(p):
    if os.path.isabs(p):
        return p
    return os.path.abspath(os.path.join(config_dir, p))
def abs_data_path(p):
    if os.path.isabs(p):
        return p
    return os.path.abspath(os.path.join(data_dir, p))

In [ ]:
def load_yamls_from_dir(dir_path):
    yamls = dict()
    for p in glob.glob(os.path.join(dir_path, "**/*.yaml"), recursive=True):
        with open(p, "r") as f:
            yamls[os.path.relpath(p, dir_path)] = yaml.safe_load(f)
    return yamls

In [ ]:
@cache
def load_patches_jsons(json_path):
    with open(json_path, "r") as f:
        data = json.load(f)
    return {patch["patchpath"] for patch in data["patches"]}

In [ ]:

def get_group_lens(config):
    group1 = load_patches_jsons(abs_data_path(config["data"]["group1_dir"]))
    group2 = load_patches_jsons(abs_data_path(config["data"]["group2_dir"]))
    dilution = load_patches_jsons(abs_data_path(config["data"]["dilution_dataset"])) if "dilution_dataset" in config["data"] else []
    overlap_group_1_2 = group1 & group2
    len_A = len(group1 - overlap_group_1_2) if config["data"]["remove_overlaps_A"] else len(group1)
    len_B = len(group2 - overlap_group_1_2) if config["data"]["remove_overlaps_B"] else len(group2)
    len_dilution = len(dilution - (group1 if config["data"]["remove_overlaps_dilution_A"] else set()) - (group2 if config["data"]["remove_overlaps_dilution_B"] else set()))
    return {"A": len_A, "B": len_B, "dilution": len_dilution}


In [ ]:
def min_achievable_concentration(len_orig_set, len_dilution_set):
    return len_orig_set / (len_orig_set + len_dilution_set)

def required_orig_set_subsampling_size(target_concentration, len_dilution_set):
    return int((target_concentration * len_dilution_set) / (1 - target_concentration))

def get_suitable_subset_size_for_target_concentration(config, target_concentration):
    group_lens = get_group_lens(config)
    set_len = min(group_lens["A"], group_lens["B"])
    if target_concentration == 1.0:
        return set_len
    if "dilution_dataset" not in config["data"]:
        return float("inf")
    return min(required_orig_set_subsampling_size(target_concentration, group_lens["dilution"]//2), set_len) # //2 because dilution images are split between both groups


def deduplicate_for_not_diluted_sets(configs):
    # Deduplicate datasets that use the same two original sets and differ only in dilution dataset
    seen = set()
    deduplicated_configs = dict()
    for name, cfg in configs.items():
        key = (cfg["data"]["group1_dir"], cfg["data"]["group2_dir"])
        if key not in seen:
            deduplicated_configs[name] = cfg
            seen.add(key)
    return deduplicated_configs

def group_same_group1_group2(configs):
    # Group datasets that use the same two original sets (regardless of dilution dataset, but keep order of sets), assign same_diff_group_name
    for cfg in configs.values():
        cfg["data"]["same_diff_group_name"] = "{0} / {1}".format(cfg['data']['group1'], cfg['data']['group2'])
    return configs

def align_config_groups_lens(suitable_configs_name_to_size, configs):
    # For each group (pair of original sets), find the minimum suitable size among all configs using that group (regardless of dilution dataset and order of sets)
    group = lambda cfg: frozenset((cfg["data"]["group1_dir"], cfg["data"]["group2_dir"]))
    groups = {group(cfg): float("inf") for cfg in configs.values()}
    groups = reduce(lambda d, name_val: {**d, group(configs[name_val[0]]): min(d[group(configs[name_val[0]])], name_val[1]["size"])}, suitable_configs_name_to_size.items(), groups)
    return {name: {**suitable_configs_name_to_size[name], "size": groups[group(configs[name])]} for name in suitable_configs_name_to_size}

# relative error of concentration due to rounding for min_num_orig_samples_in_diluted_set=3 at most ca. 16% (2 results in a max rounding error of ca. 25%, 5 results in ca. 10%)
def get_dataset_for_target_concentration(base_dir, target_concentration, min_num_orig_samples_in_diluted_set=10, deduplicate_not_diluted_sets=False):
    configs = load_yamls_from_dir(base_dir)
    configs = group_same_group1_group2(configs)
    if deduplicate_not_diluted_sets:
        assert target_concentration == 1.0, "Deduplication for not diluted sets only makes sense for target concentration 1.0"
        configs = deduplicate_for_not_diluted_sets(configs)
    config_name_to_groupinfo = {name: {"size": get_suitable_subset_size_for_target_concentration(cfg, target_concentration), "supergroup_name": "{0} / {1}".format(*sorted((cfg['data']['group1'], cfg['data']['group2']))), "same_diff_group_name": cfg["data"]["same_diff_group_name"]} for name, cfg in configs.items()}
    print(f"{config_name_to_groupinfo=}")
    suitable_configs_name_to_size = {name: info for name, info in config_name_to_groupinfo.items() if info["size"] >= min_num_orig_samples_in_diluted_set}
    return align_config_groups_lens(suitable_configs_name_to_size, configs)


In [ ]:
def write_configs_to_dir(configs_name_to_size, source_dir, target_dir, target_concentration):
    os.makedirs(target_dir, exist_ok=True)
    for name in configs_name_to_size.keys():
        source_path = os.path.join(source_dir, name)
        target_path = os.path.join(target_dir, name)
        os.makedirs(os.path.dirname(target_path), exist_ok=True)
        with open(source_path, "r") as f:
            config = yaml.safe_load(f)
        config["data"]["supergroup_name"] = configs_name_to_size[name]["supergroup_name"]
        config["data"]["same_diff_group_name"] = configs_name_to_size[name]["same_diff_group_name"]
        config["data"]["num_images_per_group"] = configs_name_to_size[name]["size"]
        config["data"]["min_concentration"] = target_concentration
        with open(target_path, "w") as f:
            yaml.safe_dump(config, f)

In [ ]:
base_dir = abs_config_path("metadata_filtered/splits_with_ground_truth/raw")

In [ ]:
no_dilution_sets = get_dataset_for_target_concentration(base_dir, target_concentration=1.)
concentration_0_125_sets = get_dataset_for_target_concentration(base_dir, target_concentration=0.125)
concentration_0_01_sets = get_dataset_for_target_concentration(base_dir, target_concentration=0.01)
concentration_0_001_sets = get_dataset_for_target_concentration(base_dir, target_concentration=0.001)
concentration_0_0001_sets = get_dataset_for_target_concentration(base_dir, target_concentration=0.0001)
print(f"Datasets suitable for no dilution (concentration=1.0): {len(no_dilution_sets)}")
print(f"Datasets suitable for concentration=0.125: {len(concentration_0_125_sets)}")
print(f"Datasets suitable for concentration=0.01: {len(concentration_0_01_sets)}")
print(f"Datasets suitable for concentration=0.001: {len(concentration_0_001_sets)}")
print(f"Datasets suitable for concentration=0.0001: {len(concentration_0_0001_sets)}")

In [ ]:
all_configs = set(load_yamls_from_dir(base_dir).keys())
print(f"Datasets removed from no dilution (concentration=1.0): {all_configs - set(no_dilution_sets.keys())}")
print(f"Datasets not suitable for concentration=0.125: {all_configs - set(concentration_0_125_sets.keys())}")
print(f"Datasets not suitable for concentration=0.01: {all_configs - set(concentration_0_01_sets.keys())}")
print(f"Datasets not suitable for concentration=0.001: {all_configs - set(concentration_0_001_sets.keys())}")
print(f"Datasets not suitable for concentration=0.0001: {all_configs - set(concentration_0_0001_sets.keys())}")

In [ ]:
def padded_json_pathname(original_json_path, pad_factor, pad_color, paint_bboxes=False, bbox_color=(255, 0, 0), bbox_thickness=5, source_padding_factor=0.0, source_padding_color=None, source_paint_bboxes=False, source_bbox_color=(255, 0, 0), source_bbox_thickness=5):
    def get_pad_str(pad_factor, pad_color, paint_bboxes, bbox_color, bbox_thickness):

        if pad_color is None and pad_factor == 0.0:
            return ""
        else:
            pad_color_str = f'c{pad_color[0]}-{pad_color[1]}-{pad_color[2]}' if pad_color is not None else 'clip'
            pad_factor_str = str(pad_factor).replace('.', '-')
            if paint_bboxes:
                bbox_color_str = f"{bbox_color[0]}-{bbox_color[1]}-{bbox_color[2]}"
                bbox_str = f"_bbox{bbox_color_str}_th{bbox_thickness}"
            else:
                bbox_str = ""
            return f"_pad{pad_factor_str}_{pad_color_str}{bbox_str}"
    base, ext = os.path.splitext(original_json_path)
    source_pad_str = get_pad_str(source_padding_factor, source_padding_color, source_paint_bboxes, source_bbox_color, source_bbox_thickness)
    if source_pad_str:
        base = os.path.join(os.path.split(base)[0], os.path.basename(base).replace(source_pad_str, ""))
    pad_str = get_pad_str(pad_factor, pad_color, paint_bboxes, bbox_color, bbox_thickness)
    base = os.path.join(os.path.split(base)[0].replace(f"/images{source_pad_str}/", f"/images{pad_str}/"), os.path.basename(base))
    return f"{base}{pad_str}{ext}"

In [ ]:
def copy_configs_for_padding_variants(source_dir, target_dir, padding_factors, padding_colors, paint_bboxes=False, bbox_color=(255, 0, 0), bbox_thickness=5, source_padding_factor=0.0, source_padding_color=None, source_paint_bboxes=False, source_bbox_color=(255, 0, 0), source_bbox_thickness=5, dry_run=False):

    configs = load_yamls_from_dir(source_dir)
    for name, cfg in configs.items():
        for pad_factor in padding_factors:
            for pad_color in padding_colors:
                if pad_factor == 0.0 and pad_color is None and not paint_bboxes:
                    target_path = os.path.join(target_dir, name)
                else:
                    target_path = os.path.join(
                        target_dir,
                        f"pad{str(pad_factor).replace('.', '-')}" + (f"_color{'-'.join(map(str, pad_color))}" if pad_color is not None else '_clip') + (f"_bbox{bbox_color[0]}-{bbox_color[1]}-{bbox_color[2]}_thickness{bbox_thickness}" if paint_bboxes else ""),
                        name,
                    )
                os.makedirs(os.path.dirname(target_path), exist_ok=True)
                cfg_copy = cfg.copy()
                cfg_copy["data"]["patch_pad_factor"] = pad_factor
                cfg_copy["data"]["patch_fill_color"] = pad_color
                # Keep behaviour identical when paint_bboxes=False; only filenames change when True
                if "dilution_dataset" in cfg_copy["data"] and cfg_copy["data"]["dilution_dataset"] is not None:
                    cfg_copy["data"]["dilution_dataset"] = padded_json_pathname(
                        cfg_copy["data"]["dilution_dataset"], pad_factor, pad_color, paint_bboxes, bbox_color, bbox_thickness, source_padding_factor, source_padding_color, source_paint_bboxes, source_bbox_color, source_bbox_thickness
                    )

                else:
                    cfg_copy["data"]["dilution_dataset"] = None
                cfg_copy["data"]["group1_dir"] = padded_json_pathname(
                    cfg_copy["data"]["group1_dir"], pad_factor, pad_color, paint_bboxes, bbox_color, bbox_thickness, source_padding_factor, source_padding_color, source_paint_bboxes, source_bbox_color, source_bbox_thickness
                )
                cfg_copy["data"]["group2_dir"] = padded_json_pathname(
                    cfg_copy["data"]["group2_dir"], pad_factor, pad_color, paint_bboxes, bbox_color, bbox_thickness, source_padding_factor, source_padding_color, source_paint_bboxes, source_bbox_color, source_bbox_thickness
                )
                if dry_run:
                    print(f"{target_path=}")
                    print(f"{cfg_copy=}")
                else:
                    with open(target_path, "w") as f:
                        yaml.safe_dump(cfg_copy, f)

In [ ]:
copy_configs_for_padding_variants(
    source_dir=abs_config_path("clip_filtered/patches_with_padding/pad0-5_clip_bbox255-0-0_thickness5/"),
    target_dir=abs_config_path("clip_filtered/patches_without_padding/"),
    padding_factors=[0.],
    padding_colors=[None],
    paint_bboxes=False,
    bbox_color=(255, 0, 0),
    bbox_thickness=5,
    source_padding_factor=0.5,
    source_bbox_color=(255, 0, 0),
    source_bbox_thickness=5,
    source_padding_color=None,
    source_paint_bboxes=True,
    dry_run=False,
)

In [ ]:
def convert_metadata_filtered_sets_for_padding_variants(source_dir, target_dir, padding_factors, padding_colors, paint_bboxes=False, bbox_color=(255, 0, 0), bbox_thickness=5,source_padding_factor=0.0, source_padding_color=None, source_paint_bboxes=False, source_bbox_color=(255, 0, 0), source_bbox_thickness=5, dry_run=False, set_patch_data_to_none_for_unpadded_targets=False, base_dir=None):
    if base_dir is None:
        base_dir = source_dir

    def padded_image_pathname(original_image_path, pad_factor, pad_color, paint_bboxes, bbox_color, bbox_thickness, source_padding_factor, source_padding_color, source_paint_bboxes, source_bbox_color, source_bbox_thickness):
        def get_pad_str(pad_factor, pad_color, paint_bboxes, bbox_color, bbox_thickness):
            if pad_color is None and pad_factor == 0.0:
                return "images"
            else:
                return f"images_pad{str(pad_factor).replace('.', '-')}_{'c' + '-'.join(map(str, pad_color)) if pad_color is not None else 'clip'}{'_bbox' + '-'.join(map(str, bbox_color)) + '_th' + str(bbox_thickness) if paint_bboxes else ''}"
        source_padding_str = get_pad_str(source_padding_factor, source_padding_color, source_paint_bboxes, source_bbox_color, source_bbox_thickness)
        pad_str = get_pad_str(pad_factor, pad_color, paint_bboxes, bbox_color, bbox_thickness)
        return original_image_path.replace(f"/{source_padding_str}/", f"/{pad_str}/")


    jsons = glob.glob(os.path.join(source_dir, "**/*.json"), recursive=True)
    # only keep jsons that correspond to the source padding variant (in case there are jsons for multiple padding variants in the source dir)
    source_pad_str = f"_pad{str(source_padding_factor).replace('.', '-')}" + (f"_c{source_padding_color[0]}-{source_padding_color[1]}-{source_padding_color[2]}" if source_padding_color is not None else "_clip") + (f"_bbox{source_bbox_color[0]}-{source_bbox_color[1]}-{source_bbox_color[2]}_th{source_bbox_thickness}" if source_paint_bboxes else "")
    jsons = [j for j in jsons if source_pad_str in j or (source_pad_str == "_pad0" and "/images/" in j)] # if source padding is unpadded, look for jsons with "/images/" in the path (assuming that all unpadded jsons are in an "images" folder and all padded variants are in folders with "pad{pad_factor}" in the name)
    for json_path in tqdm.tqdm(jsons, desc="Converting metadata filtered jsons for padding variants"):
        print(f"Processing {json_path}...")
        with open(json_path, "r") as f:
            data = json.load(f)
        relative_json_path = os.path.relpath(json_path, base_dir)
        for pad_factor in padding_factors:
            for pad_color in padding_colors:
                data_copy = data.copy()
                data_copy["padding_factor"] = pad_factor
                data_copy["padding_color"] = pad_color
                data_copy["bbox_color"] = bbox_color if paint_bboxes else None
                data_copy["bbox_thickness"] = bbox_thickness if paint_bboxes else None
                for patch in data_copy["patches"]:
                    patch["patchpath"] = padded_image_pathname(patch["patchpath"], pad_factor, pad_color, paint_bboxes, bbox_color, bbox_thickness, source_padding_factor, source_padding_color, source_paint_bboxes, source_bbox_color, source_bbox_thickness)
                    if pad_factor != source_padding_factor or sum(1 for v in [pad_color, source_padding_color] if v is None) == 1: # if pad_color xor source_padding_color is None, one of them is centered and the other is not => patch origin might be different
                        # look up new patch dimensions from extracted_patches
                        pad_str = os.path.basename(os.path.dirname(patch["patchpath"])).replace("images", "")
                        if set_patch_data_to_none_for_unpadded_targets and pad_str == "": # avoid time-consuming lookup for unpadded targets if we know that patch_data should be set to None for them
                            patch["patch_data"]["patch_width"] = None
                            patch["patch_data"]["patch_height"] = None
                            patch["patch_data"]["patch_top_left"] = None
                        else:
                            extracted_patches_jsons = glob.glob(os.path.join(os.path.join(os.path.dirname(abs_data_path(patch["patchpath"])), os.pardir), f"*{pad_str}.json"))
                            if not pad_str: # if pad_str is empty (unpadded target) look for extracted_patches without "_pad" in the name
                                extracted_patches_jsons = [p for p in extracted_patches_jsons if "_pad" not in os.path.basename(p)]
                            assert len(extracted_patches_jsons) == 1, f"Expected exactly one extracted_patches json for pad_str {pad_str} in {os.path.join(os.path.dirname(abs_data_path(patch['patchpath'])), os.pardir)}, but found {len(extracted_patches_jsons)}: {extracted_patches_jsons}"
                            with open(extracted_patches_jsons[0], "r") as f:
                                extracted_patches_data = json.load(f)
                            matching_patch = next((p for p in extracted_patches_data["patches"] if p["patchpath"] == patch["patchpath"]), None)
                            if matching_patch is not None:
                                patch["patch_data"]["patch_width"] = matching_patch["patch_data"].get("patch_width")
                                patch["patch_data"]["patch_height"] = matching_patch["patch_data"].get("patch_height")
                                patch["patch_data"]["patch_top_left"] = matching_patch["patch_data"].get("patch_top_left")
                            else:
                                raise ValueError(f"Could not find matching patch for {patch['patchpath']} in extracted_patches json {extracted_patches_jsons[0]}")
                target_json_path = padded_json_pathname(os.path.join(target_dir, relative_json_path), pad_factor, pad_color, paint_bboxes=paint_bboxes, bbox_color=bbox_color, bbox_thickness=bbox_thickness, source_padding_factor=source_padding_factor, source_padding_color=source_padding_color, source_paint_bboxes=source_paint_bboxes, source_bbox_color=source_bbox_color, source_bbox_thickness=source_bbox_thickness)

                if not dry_run:
                    os.makedirs(os.path.dirname(target_json_path), exist_ok=True)
                    with open(target_json_path, "w") as f:
                        json.dump(data_copy, f, indent=4)
                else:
                    print(f"{target_json_path=}")
                    print(f"{data_copy=}")




In [ ]:
convert_metadata_filtered_sets_for_padding_variants(
    source_dir=abs_data_path("clip_filtered/waymo/train/vehicle/"),
    target_dir=abs_data_path("clip_filtered"),
    padding_factors=[0.],
    padding_colors=[None],
    paint_bboxes=False,
    bbox_color=(255, 0, 0),
    bbox_thickness=5,
    source_padding_factor=0.5,
    source_padding_color=None,
    source_paint_bboxes=True,
    source_bbox_color=(255, 0, 0),
    source_bbox_thickness=5,
    dry_run=False,
    set_patch_data_to_none_for_unpadded_targets=False,
    base_dir=abs_data_path("clip_filtered")
)

In [ ]:
def convert_dirs_config_to_jsons_config(source_dir, remove_overlaps_A=False, remove_overlaps_B=False, remove_overlaps_dilution_A=False, remove_overlaps_dilution_B=False, dry_run=False):
    configs = load_yamls_from_dir(source_dir)
    for name, cfg in configs.items():
        assert cfg["data"]["mode"] == "dirs", f"Config {name} is not in 'dirs' mode, cannot convert to 'jsons' config"
        cfg_copy = cfg.copy()
        cfg_copy["data"]["remove_overlaps_A"] = remove_overlaps_A
        cfg_copy["data"]["remove_overlaps_B"] = remove_overlaps_B
        cfg_copy["data"]["remove_overlaps_dilution_A"] = remove_overlaps_dilution_A
        cfg_copy["data"]["remove_overlaps_dilution_B"] = remove_overlaps_dilution_B
        cfg_copy["data"]["min_concentration"] = 1.0
        cfg_copy["data"]["mode"] = "jsons"
        jsons1 = glob.glob(os.path.join(abs_data_path(cfg["data"]["group1_dir"]), "*.json"))
        jsons2 = glob.glob(os.path.join(abs_data_path(cfg["data"]["group2_dir"]), "*.json"))
        assert len(jsons1) == 1, f"Expected exactly one json file in group1_dir {cfg['data']['group1_dir']}, found {len(jsons1)}"
        cfg_copy["data"]["group1_dir"] = os.path.relpath(jsons1[0], data_dir)
        if len(jsons2) == 0:
            pad_str = os.path.basename(cfg["data"]["group2_dir"]).replace("images", "")
            jsons2 = glob.glob(os.path.join(os.path.join(abs_data_path(cfg["data"]["group2_dir"]), os.pardir), f"*{pad_str}.json"))
            if not pad_str: # if pad_str is empty (unpadded target) look for extracted_patches without "_pad" in the name
                jsons2 = [p for p in jsons2 if "_pad" not in os.path.basename(p)]
            assert len(jsons2) == 1, f"Expected exactly one json file in group2_dir {cfg['data']['group2_dir']}, found {len(jsons2)}"
        cfg_copy["data"]["group2_dir"] = os.path.relpath(jsons2[0], data_dir)
        if dry_run:
            print(f"Would convert config {name} to 'jsons' mode with the following content:")
            print(yaml.safe_dump(cfg_copy))
        else:
            with open(os.path.join(source_dir, name), "w") as f:
                yaml.safe_dump(cfg_copy, f)

In [ ]:
convert_dirs_config_to_jsons_config(abs_config_path("clip_filtered/patches_with_padding/pad0-5_clip_bbox255-0-0_thickness5/"), dry_run=True)

In [ ]:
def check_and_fix_patch_dimensions_in_json(json_path, dry_run=False):
    @cache
    def cached_load_json(json_path):
        with open(json_path, "r") as f:
            return json.load(f)

    def load_extracted_patches_jsons_for_patch(patch):
        pad_str = os.path.basename(os.path.dirname(abs_data_path(patch["patchpath"]))).replace("images", "")
        extracted_patches_jsons = glob.glob(os.path.join(os.path.join(os.path.dirname(abs_data_path(patch["patchpath"])), os.pardir), f"*{pad_str}.json"))
        if not pad_str: # if pad_str is empty (unpadded target) look for extracted_patches without "_pad" in the name
            extracted_patches_jsons = [p for p in extracted_patches_jsons if "_pad" not in os.path.basename(p)]
        assert len(extracted_patches_jsons) == 1, f"Expected exactly one extracted_patches json for pad_str {pad_str} in {os.path.join(os.path.dirname(abs_data_path(patch['patchpath'])), os.pardir)}, but found {len(extracted_patches_jsons)}: {extracted_patches_jsons}"
        return cached_load_json(extracted_patches_jsons[0])

    with open(json_path, "r") as f:
        data = json.load(f)
    if len(data["patches"]) == 0:
        print(f"Warning: No patches found in json file {json_path}, skipping patch dimension check")
        return
    extracted_patches_data = load_extracted_patches_jsons_for_patch(data["patches"][0])
    modified = False
    for patch in data["patches"]:
        patchpath = patch["patchpath"]
        extracted_patches_index = int(patchpath[patchpath.rindex("_") + 1:patchpath.rindex(".")])
        probable_matching_patchpath = extracted_patches_data["patches"][extracted_patches_index]["patchpath"]
        if probable_matching_patchpath != patchpath:
            matching_patch = next((p for p in extracted_patches_data["patches"] if p["patchpath"] == patchpath), None)
            print(f"Warning: Expected patchpath {patchpath} at index {extracted_patches_index} in extracted_patches json {extracted_patches_data}, but found {probable_matching_patchpath}. Looking up patchpath in extracted_patches json...")
        else:
            matching_patch = extracted_patches_data["patches"][extracted_patches_index]
        if matching_patch is not None:
            if patch["patch_data"].get("patch_width") != matching_patch["patch_data"].get("patch_width") or patch["patch_data"].get("patch_height") != matching_patch["patch_data"].get("patch_height") or patch["patch_data"].get("patch_top_left") != matching_patch["patch_data"].get("patch_top_left"):
                modified = True
                if dry_run:
                    print(f"Would update patch_width from {patch['patch_data'].get('patch_width')} to {matching_patch['patch_data'].get('patch_width')}, patch_height from {patch['patch_data'].get('patch_height')} to {matching_patch['patch_data'].get('patch_height')}, patch_top_left from {patch['patch_data'].get('patch_top_left')} to {matching_patch['patch_data'].get('patch_top_left')} for patchpath {patchpath} in json file {json_path}")
                    break # only print the first mismatch for brevity
            patch["patch_data"]["patch_width"] = matching_patch["patch_data"].get("patch_width")
            patch["patch_data"]["patch_height"] = matching_patch["patch_data"].get("patch_height")
            patch["patch_data"]["patch_top_left"] = matching_patch["patch_data"].get("patch_top_left")
        else:
            raise ValueError(f"Could not find matching patch for {patch['patchpath']} in extracted_patches json {extracted_patches_data}")
    if modified:
        if not dry_run:
            with open(json_path, "w") as f:
                json.dump(data, f, indent=4)
        else:
            print(f"Would update json file {json_path} with fixed patch dimensions")


In [ ]:
def check_and_fix_patch_dimensions_in_jsons(json_dir, dry_run=False):
    jsons = glob.glob(os.path.join(json_dir, "**/*.json"), recursive=True)
    for json_path in tqdm.tqdm(jsons, desc=f"Checking {'and fixing ' if not dry_run else ''}patch dimensions in jsons"):
        check_and_fix_patch_dimensions_in_json(json_path, dry_run=dry_run)

In [ ]:
check_and_fix_patch_dimensions_in_jsons(abs_data_path("clip_filtered"), dry_run=False)

In [ ]:
check_and_fix_patch_dimensions_in_jsons(abs_data_path("metadata_filtered/splits_without_ground_truth"), dry_run=False)